### Importing Libraries

In [ ]:
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn

### Loading Datasets

In [ ]:
state_df = pd.read_csv('/observation_new')
action_df = pd.read_csv('/action_new')

In [ ]:
state_df.head()

,state_0,state_1,state_2,state_3,state_4,state_5,state_6,state_7,state_8,state_9,...,state_21,state_22,state_23,state_24,state_25,state_26,state_27,state_28,state_29,state_30
0,0.018257,0.018257,-0.000016,-0.001022,-0.600331,-0.378351,0.468709,-1.798005,1.798133,0.002816,...,0.792600,-0.793677,-0.000003,-6.297545e-06,0.000004,-4.185603e-10,-0.794343,0.793510,-1.191359e-06,-0.000005
1,0.011092,0.011092,0.000706,-0.001152,-0.600212,-0.379343,0.466962,-1.796333,1.796523,0.004250,...,0.799863,-0.799950,-0.000001,-3.898578e-06,0.000002,-1.989670e-10,-0.800120,0.799998,4.612901e-07,-0.000003
2,0.006739,0.006739,0.001295,-0.001390,-0.600143,-0.380520,0.465257,-1.794547,1.795105,0.005723,...,0.799929,-0.800000,-0.000001,-3.357395e-06,0.000002,-2.015183e-10,-0.800067,0.800042,5.632125e-07,-0.000002
3,0.004094,0.004094,0.001788,-0.001716,-0.600101,-0.382145,0.463310,-1.792537,1.793805,0.007541,...,0.799929,-0.800000,-0.000001,-1.948255e-06,0.000002,-1.865733e-10,-0.800063,0.800042,-9.703118e-07,-0.000002
4,0.002663,0.002663,0.002228,-0.002107,-0.600069,-0.384191,0.461054,-1.790260,1.792584,0.009705,...,0.799929,-0.800000,-0.000002,-9.284292e-07,0.000002,-1.817493e-10,-0.800062,0.800042,-2.188611e-06,-0.000002


### Spliting dataset in to train and test data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(state_df, action_df, test_size=0.3, random_state=12)

### Scaling the Dataset

In [ ]:
scaler_X = joblib.load('/scaler_X.pkl')
scaler_y = joblib.load('/scaler_y.pkl')

/usr/local/lib/python3.11/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.0.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [ ]:
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

y_train_scaled = scaler_y.fit_transform(y_train)
y_test_scaled = scaler_y.transform(y_test)

### Creating train and test data tensors

In [ ]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_scaled, dtype=torch.float32)

train_ds = TensorDataset(X_train_tensor, y_train_tensor)
test_ds = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32)

### Defining Regressor model

In [ ]:
class MultilayerRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(31, 129),
            nn.ReLU(),
            nn.Linear(129, 611),
            nn.ReLU(),
            nn.Linear(611, 21)
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
model = MultilayerRegressor()
criterian = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

### training the model

In [ ]:
num_epochs = 100

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for xb, yb in train_loader:
        pred = model(xb)
        loss = criterian(pred, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)

print(f'Epoch {epoch+1}, Train Loss: {average_loss:.4}')

Epoch 100, Train Loss: 0.04789


### Evaluating model

In [ ]:
model.eval()
with torch.no_grad():
    prediction = model(X_test_tensor)
    test_loss = criterian(prediction, y_test_tensor)
    print(f'Test MSE: {test_loss.item():.4f}')

Test MSE: 0.0474


In [ ]:
preds_numpy = prediction.numpy()
preds_original = scaler_y.inverse_transform(preds_numpy)

### Loading new dataset to test model

In [ ]:
state_v1 = pd.read_csv('/observation_v1')
action_v1 = pd.read_csv('/action_v1')

state_v2 = pd.read_csv('/observation_v2')
action_v2 = pd.read_csv('/action_v2')

### Scaling test data

In [ ]:
state_v1_scaled = scaler_X.transform(state_v1)
state_v2_scaled = scaler_X.transform(state_v2)

action_v1_scaled = scaler_y.transform(action_v1)
action_v2_scaled = scaler_y.transform(action_v2)

### Creating test data tensors

In [ ]:
state_v1_tensor = torch.tensor(state_v1_scaled, dtype= torch.float32)
state_v2_tensor = torch.tensor(state_v2_scaled, dtype= torch.float32)

action_v1_tensor = torch.tensor(action_v1_scaled, dtype= torch.float32)
action_v2_tensor = torch.tensor(action_v2_scaled, dtype= torch.float32)

### Model evaluation on Test data

In [ ]:
# V1 test

model.eval()
with torch.no_grad():
    prediction_v1 = model(state_v1_tensor)
    test_loss_v1 = criterian(prediction_v1, action_v1_tensor)
    print(f'Test MSE V1: {test_loss_v1.item():.4f}')

Test MSE V1: 0.6498


In [ ]:
# V2 test

model.eval()
with torch.no_grad():
    prediction_v2 = model(state_v2_tensor)
    test_loss_v2 = criterian(prediction_v2, action_v2_tensor)
    print(f'Test MSE V2: {test_loss_v2.item():.4f}')

Test MSE V2: 0.6498


### Varience for test data result

In [ ]:
variance_v1 = torch.var(action_v1_tensor, unbiased= True)
print(variance_v1)

tensor(1.2387)


In [ ]:
variance_v2 = torch.var(action_v2_tensor, unbiased= True)
print(variance_v2)

tensor(1.2387)
